# SEC EDGAR Silver Layer - Company Facts (XBRL)

Production silver build for `abfss://bronze@stdsecusa.dfs.core.windows.net/SEC_EDGAR/company_facts/` (SEC `data.sec.gov/api/xbrl/companyfacts/CIK##########.json` snapshots). Output: `sec_edgar.silver.*` stored under the `silver_usa` external location.

**Bronze findings (profiled 2026-11-15)**
* 10 files, one per CIK (`cik=<10-digit>/<10-digit>.json`), 44 MB, full snapshot per company (not paginated; each file is the complete fact history).
* Every file starts with a **UTF-8 BOM** - standard JSON parsers fail; we strip it.
* Structure: `{cik, entityName, facts: {taxonomy: {concept: {label, description, units: {unit: [fact...]}}}}}`. 6 taxonomies (`us-gaap`, `dei`, `srt`, `ecd`, `ffd`, `invest`), 2,407 concepts, 36 units, **295,698 facts**. Struct inference would create thousands of columns -> we use an explicit MAP schema to extract every taxonomy/concept/unit dynamically.
* `cik` is INT in JSON but zero-padded in the path; `val` mixes integers and decimals (max 8.08e13, 5 dp) -> parsed directly to `DECIMAL(38,6)` (no float rounding).
* `start` is absent for instant (balance-sheet) facts (41%); `frame` present on only 40% (SEC's de-duplicated canonical value).
* 56% of (cik, concept, unit, period) appear in >1 filing (prior-period comparatives / restatements) -> keep all, flag latest.
* `fy`/`fp` are the fiscal year/period **of the filing**, not of the fact (fy - year(end) ranges -1..4); null for 8-K/424B/DEF 14A etc.
* 25 facts filed before period end (424B2 forward-looking amounts, authorised-share counts) -> warning, not error.
* Unit spelling drift: `segement`, `segment`, `Segment`, `reportable_segment`; `Store`/`stores`; `Year`/`years` -> normalised column added, raw kept.

**10 steps:** 1 Config & run context | 2 DDL with contracts | 3 File manifest & incremental detection | 4 Robust parse (BOM, exact decimals, corrupt files) | 5 Schema-drift detection | 6 Full flatten & typing | 7 Data-quality rules + quarantine | 8 Deduplication, keys & business flags | 9 Idempotent MERGE (insert/update/soft-delete) | 10 Reconciliation, audit & optimisation

**Upstream (bronze) recommendations:** SEC requires a descriptive `User-Agent` and <=10 req/s; companyfacts is not paginated, but the `submissions` API is - older filings are in `filings.files[]` (CIK##########-submissions-001.json ...) and must be followed to get complete filing history. Write bronze files without BOM and keep immutable dated copies (e.g. `ingest_date=YYYY-MM-DD/`) instead of overwriting.

In [0]:
# Step 1: parameterise everything (jobs can override via widgets) and create a unique run_id for lineage/audit
import uuid, datetime, json
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable

dbutils.widgets.text("bronze_path", "abfss://bronze@stdsecusa.dfs.core.windows.net/SEC_EDGAR/company_facts")
dbutils.widgets.text("silver_location", "abfss://silver@stdsecusa.dfs.core.windows.net/SEC_EDGAR")
dbutils.widgets.text("catalog", "sec_edgar")
dbutils.widgets.text("schema", "silver")
dbutils.widgets.dropdown("full_refresh", "false", ["true", "false"])

BRONZE_PATH  = dbutils.widgets.get("bronze_path").rstrip("/")
SILVER_LOC   = dbutils.widgets.get("silver_location").rstrip("/")
CAT, SCH     = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"
NS = f"{CAT}.{SCH}"
T_FACTS, T_QUAR, T_COMPANY, T_CONCEPT = f"{NS}.company_facts", f"{NS}.company_facts_quarantine", f"{NS}.company", f"{NS}.concept"
T_MANIFEST, T_AUDIT, T_DQ, T_DRIFT = f"{NS}.ingest_manifest", f"{NS}.audit_runs", f"{NS}.dq_results", f"{NS}.schema_drift_log"

RUN_ID    = str(uuid.uuid4())
RUN_START = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)
RUN_USER  = spark.sql("select current_user()").first()[0]
PERIODIC_FORMS = ["10-K", "10-Q", "10-K/A", "10-Q/A", "20-F", "40-F", "20-F/A", "40-F/A"]
EXPECTED_FACT_KEYS = {"start", "end", "val", "accn", "fy", "fp", "form", "filed", "frame"}
EXPECTED_TOP_KEYS  = {"cik", "entityName", "facts"}
EXPECTED_CONCEPT_KEYS = {"label", "description", "units"}
print(f"run_id={RUN_ID} user={RUN_USER} full_refresh={FULL_REFRESH} target={NS}")

In [0]:
# Step 2: explicit DDL = data contract. NOT NULL + CHECK constraints, comments, CDF for downstream incremental gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NS} MANAGED LOCATION '{SILVER_LOC}' COMMENT 'SEC EDGAR silver layer (cleansed, conformed XBRL company facts)'")

TBLPROPS = "TBLPROPERTIES ('delta.enableChangeDataFeed'='true', 'delta.enableDeletionVectors'='true', 'quality'='silver')"

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_FACTS} (
  fact_id STRING NOT NULL COMMENT 'sha256 of natural key (cik|taxonomy|concept|unit|period_start|period_end|accession_number)',
  cik BIGINT NOT NULL, cik_padded STRING NOT NULL COMMENT '10-digit zero padded CIK',
  entity_name STRING, taxonomy STRING NOT NULL, concept STRING NOT NULL, concept_label STRING,
  unit STRING NOT NULL COMMENT 'unit exactly as reported', unit_normalized STRING NOT NULL,
  period_type STRING NOT NULL COMMENT 'instant | duration', period_start DATE, period_end DATE NOT NULL, duration_days INT,
  value DECIMAL(38,6) NOT NULL, accession_number STRING NOT NULL, form STRING NOT NULL, is_amendment BOOLEAN NOT NULL,
  filing_fiscal_year INT COMMENT 'fy of the FILING, not of the fact period', filing_fiscal_period STRING COMMENT 'fp of the FILING',
  filed_date DATE NOT NULL, frame STRING COMMENT 'SEC calendar frame; set only on the de-duplicated canonical fact',
  is_latest_filing BOOLEAN NOT NULL COMMENT 'most recently filed value for this cik/concept/unit/period',
  dq_warnings ARRAY<STRING>, record_hash STRING NOT NULL,
  source_file STRING NOT NULL, source_file_sha256 STRING NOT NULL, source_file_mtime TIMESTAMP,
  is_active BOOLEAN NOT NULL COMMENT 'false = no longer present in latest bronze snapshot (soft delete)',
  first_loaded_run_id STRING NOT NULL, last_updated_run_id STRING NOT NULL, _created_at TIMESTAMP NOT NULL, _updated_at TIMESTAMP NOT NULL,
  CONSTRAINT pk_fact PRIMARY KEY (fact_id) RELY
) CLUSTER BY (cik, concept, period_end) COMMENT 'One row per XBRL fact per filing (all taxonomies, concepts and units)' {TBLPROPS}""")

# CHECK constraints are enforced by Delta on every write (added idempotently)
existing = {k.lower() for k in spark.sql(f"SHOW TBLPROPERTIES {T_FACTS}").toPandas().key if k.startswith("delta.constraints.")}
for name, expr in {"chk_period": "period_start IS NULL OR period_start <= period_end",
                   "chk_ptype": "period_type IN ('instant','duration')",
                   "chk_accn": "accession_number RLIKE '^[0-9]{10}-[0-9]{2}-[0-9]{6}$'"}.items():
    if f"delta.constraints.{name}" not in existing:
        spark.sql(f"ALTER TABLE {T_FACTS} ADD CONSTRAINT {name} CHECK ({expr})")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_QUAR} (
  run_id STRING, source_file STRING, cik BIGINT, taxonomy STRING, concept STRING, unit STRING,
  raw_fact STRING COMMENT 'original fact as JSON', dq_errors ARRAY<STRING>, quarantined_at TIMESTAMP
) COMMENT 'Facts rejected by hard DQ rules - reviewed, never silently dropped'""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_COMPANY} (
  cik BIGINT NOT NULL, cik_padded STRING NOT NULL, entity_name STRING NOT NULL,
  taxonomies ARRAY<STRING>, concept_count INT, fact_count BIGINT, first_period_end DATE, last_period_end DATE, last_filed_date DATE,
  source_file STRING, source_file_sha256 STRING, last_updated_run_id STRING, _updated_at TIMESTAMP,
  CONSTRAINT pk_company PRIMARY KEY (cik) RELY
) COMMENT 'Company dimension (SCD1) derived from companyfacts' {TBLPROPS}""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_CONCEPT} (
  taxonomy STRING NOT NULL, concept STRING NOT NULL, label STRING, description STRING, units ARRAY<STRING>,
  last_updated_run_id STRING, _updated_at TIMESTAMP,
  CONSTRAINT pk_concept PRIMARY KEY (taxonomy, concept) RELY
) COMMENT 'XBRL concept dictionary (SCD1)' {TBLPROPS}""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_MANIFEST} (
  source_file STRING NOT NULL, file_size BIGINT, file_mtime TIMESTAMP, sha256 STRING, cik_from_path STRING,
  status STRING COMMENT 'PROCESSED | SKIPPED_UNCHANGED | FAILED_PARSE', fact_count BIGINT, run_id STRING, processed_at TIMESTAMP
) COMMENT 'Per-file ingestion ledger: drives incremental loads and file-level lineage'""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_AUDIT} (
  run_id STRING NOT NULL, run_start TIMESTAMP, run_end TIMESTAMP, status STRING, run_user STRING, full_refresh BOOLEAN,
  files_discovered INT, files_processed INT, files_skipped INT, files_failed INT,
  rows_bronze BIGINT, rows_valid BIGINT, rows_quarantined BIGINT, rows_duplicates_removed BIGINT,
  rows_inserted BIGINT, rows_updated BIGINT, rows_soft_deleted BIGINT, reconciliation_passed BOOLEAN, error_message STRING
) COMMENT 'One row per silver run'""")

spark.sql(f"CREATE TABLE IF NOT EXISTS {T_DQ} (run_id STRING, rule STRING, severity STRING, failed_rows BIGINT, total_rows BIGINT, checked_at TIMESTAMP) COMMENT 'DQ rule results per run'")
spark.sql(f"CREATE TABLE IF NOT EXISTS {T_DRIFT} (run_id STRING, level STRING, unexpected_key STRING, occurrences BIGINT, detected_at TIMESTAMP) COMMENT 'Unknown JSON keys detected in bronze'")

# runs that died mid-way (cluster loss, cancel) stay RUNNING - mark them ABANDONED so monitoring sees them
spark.sql(f"UPDATE {T_AUDIT} SET status = 'ABANDONED', run_end = current_timestamp() WHERE status = 'RUNNING' AND run_start < current_timestamp() - INTERVAL 1 MINUTE")
# open the audit record immediately so failed runs are visible too
spark.sql(f"INSERT INTO {T_AUDIT} (run_id, run_start, status, run_user, full_refresh) VALUES ('{RUN_ID}', '{RUN_START}', 'RUNNING', '{RUN_USER}', {FULL_REFRESH})")
display(spark.sql(f"SHOW TABLES IN {NS}"))

In [0]:
# Step 3: inventory every bronze file recursively; process only new/changed files (by content hash) unless full_refresh
def list_recursive(path):
    out = []
    for f in dbutils.fs.ls(path):
        out.extend(list_recursive(f.path) if f.isDir() else [(f.path, f.size, f.modificationTime)])
    return out

inventory = [r for r in list_recursive(BRONZE_PATH) if r[0].lower().endswith(".json")]
FILES_DISCOVERED = len(inventory)
assert FILES_DISCOVERED > 0, f"No JSON files found under {BRONZE_PATH}"

# read RAW BYTES (binaryFile): the text reader silently strips the BOM, hiding the defect. sha256 = true file hash.
raw_df = (spark.read.format("binaryFile").load([p for p, _, _ in inventory])
          .select(F.col("path").alias("source_file"),
                  F.col("length").alias("file_size"),
                  F.col("modificationTime").alias("file_mtime"),
                  F.regexp_extract("path", r"cik=(\d{10})", 1).alias("cik_from_path"),
                  F.sha2(F.col("content"), 256).alias("sha256"),
                  (F.hex(F.substring("content", 1, 3)) == "EFBBBF").alias("had_bom"),
                  F.decode("content", "UTF-8").alias("raw")))

last_ok = (spark.table(T_MANIFEST).filter("status = 'PROCESSED'")
           .withColumn("rn", F.row_number().over(Window.partitionBy("source_file").orderBy(F.desc("processed_at"))))
           .filter("rn = 1").select("source_file", F.col("sha256").alias("prev_sha")))
raw_df = raw_df.join(last_ok, "source_file", "left")
to_process = raw_df if FULL_REFRESH else raw_df.filter("prev_sha IS NULL OR prev_sha <> sha256")
skipped = raw_df.filter("prev_sha IS NOT NULL AND prev_sha = sha256").select("source_file", "file_size", "file_mtime", "sha256", "cik_from_path")

FILES_SKIPPED = 0 if FULL_REFRESH else skipped.count()
FILES_TO_PROCESS = FILES_DISCOVERED - FILES_SKIPPED
print(f"discovered={FILES_DISCOVERED} to_process={FILES_TO_PROCESS} skipped_unchanged={FILES_SKIPPED}")
display(raw_df.drop("raw"))

# short-circuit: nothing changed -> log skipped files, close audit, exit cleanly (re-runs are true no-ops)
if FILES_TO_PROCESS == 0:
    (skipped.withColumn("status", F.lit("SKIPPED_UNCHANGED")).withColumn("fact_count", F.lit(None).cast("bigint"))
            .withColumn("run_id", F.lit(RUN_ID)).withColumn("processed_at", F.current_timestamp())
            .select("source_file", "file_size", "file_mtime", "sha256", "cik_from_path", "status", "fact_count", "run_id", "processed_at")
            .write.mode("append").saveAsTable(T_MANIFEST))
    spark.sql(f"""UPDATE {T_AUDIT} SET run_end = current_timestamp(), status = 'NO_CHANGES', files_discovered = {FILES_DISCOVERED},
      files_processed = 0, files_skipped = {FILES_SKIPPED}, files_failed = 0, rows_bronze = 0, rows_valid = 0, rows_quarantined = 0,
      rows_duplicates_removed = 0, rows_inserted = 0, rows_updated = 0, rows_soft_deleted = 0, reconciliation_passed = true
      WHERE run_id = '{RUN_ID}'""")
    dbutils.notebook.exit(json.dumps({"run_id": RUN_ID, "status": "NO_CHANGES", "files_skipped": FILES_SKIPPED}))

In [0]:
# Step 4: explicit MAP-based schema extracts ALL taxonomies/concepts/units without inference or column explosion.
# val -> DECIMAL(38,6) straight from JSON text (no double rounding). BOM stripped before parsing.
fact_t = T.StructType([T.StructField("start", T.StringType()), T.StructField("end", T.StringType()),
                       T.StructField("val", T.DecimalType(38, 6)), T.StructField("accn", T.StringType()),
                       T.StructField("fy", T.IntegerType()), T.StructField("fp", T.StringType()),
                       T.StructField("form", T.StringType()), T.StructField("filed", T.StringType()),
                       T.StructField("frame", T.StringType())])
concept_t = T.StructType([T.StructField("label", T.StringType()), T.StructField("description", T.StringType()),
                          T.StructField("units", T.MapType(T.StringType(), T.ArrayType(fact_t)))])
doc_t = T.StructType([T.StructField("cik", T.LongType()), T.StructField("entityName", T.StringType()),
                      T.StructField("facts", T.MapType(T.StringType(), T.MapType(T.StringType(), concept_t)))])

parsed = (to_process
          .withColumn("json", F.regexp_replace("raw", "^\uFEFF", ""))
          .withColumn("doc", F.from_json("json", doc_t, {"mode": "PERMISSIVE"}))
          .withColumn("parse_error", F.when(F.col("doc").isNull() | F.col("doc.cik").isNull() | F.col("doc.facts").isNull(), "UNPARSEABLE_OR_MISSING_ROOT_KEYS")
                                     .when(F.lpad(F.col("doc.cik").cast("string"), 10, "0") != F.col("cik_from_path"), "CIK_MISMATCH_PATH_VS_PAYLOAD")))

file_status = parsed.select("source_file", "file_size", "file_mtime", "sha256", "cik_from_path", "had_bom", "parse_error", F.col("doc.entityName").alias("entity_name")).toPandas()
bad_files = file_status[file_status.parse_error.notna()]
FILES_FAILED = len(bad_files)
good = parsed.filter("parse_error IS NULL")
print(f"files with BOM: {int(file_status.had_bom.sum())}/{len(file_status)}  failed parse: {FILES_FAILED}")
if FILES_FAILED:
    display(bad_files)  # corrupt files are logged in manifest as FAILED_PARSE, never silently ignored
display(file_status)

In [0]:
# Step 5: an explicit schema silently ignores NEW fields. Detect any unknown keys at every nesting level and log them.
fact_keys_t = T.MapType(T.StringType(), T.MapType(T.StringType(), T.StructType([
    T.StructField("units", T.MapType(T.StringType(), T.ArrayType(T.MapType(T.StringType(), T.StringType()))))]))) 
concept_keys_t = T.MapType(T.StringType(), T.MapType(T.StringType(), T.MapType(T.StringType(), T.StringType())))

drift_src = good.select(F.from_json("json", T.MapType(T.StringType(), T.StringType())).alias("top"),
                        F.from_json(F.get_json_object("json", "$.facts"), concept_keys_t).alias("c"),
                        F.from_json(F.get_json_object("json", "$.facts"), fact_keys_t).alias("f"))
top_keys  = drift_src.select(F.explode(F.map_keys("top")).alias("k")).withColumn("level", F.lit("root"))
conc_keys = (drift_src.select(F.explode("c").alias("t", "cs")).select(F.explode("cs").alias("n", "cv"))
             .select(F.explode(F.map_keys("cv")).alias("k")).withColumn("level", F.lit("concept")))
fact_keys = (drift_src.select(F.explode("f").alias("t", "cs")).select(F.explode("cs").alias("n", "cv"))
             .select(F.explode("cv.units").alias("u", "arr")).select(F.explode("arr").alias("e"))
             .select(F.explode(F.map_keys("e")).alias("k")).withColumn("level", F.lit("fact")))
expected = {"root": EXPECTED_TOP_KEYS, "concept": EXPECTED_CONCEPT_KEYS, "fact": EXPECTED_FACT_KEYS}
observed = top_keys.unionByName(conc_keys).unionByName(fact_keys).groupBy("level", "k").count().toPandas()
drift = observed[[k not in expected[l] for l, k in zip(observed.level, observed.k)]]
if len(drift):
    (spark.createDataFrame(drift.rename(columns={"k": "unexpected_key", "count": "occurrences"}))
         .withColumn("run_id", F.lit(RUN_ID)).withColumn("detected_at", F.current_timestamp())
         .select("run_id", "level", "unexpected_key", "occurrences", "detected_at").write.mode("append").saveAsTable(T_DRIFT))
    print("WARNING schema drift - new keys not yet mapped to silver:"); display(drift)
else:
    print("No schema drift. Observed keys:"); display(observed.sort_values(["level", "k"]))

In [0]:
# Step 6: explode file -> taxonomy -> concept -> unit -> fact. Cast strings to proper types with try_* (never fail the job on one bad value).
UNIT_MAP = {"segement": "segment", "Segment": "segment", "reportable_segment": "segment", "stores": "Store",
            "years": "Year", "USD/shares_unit": "USD/shares"}
unit_map_expr = F.create_map(*[F.lit(x) for kv in UNIT_MAP.items() for x in kv])

concepts_df = (good.select("source_file", "sha256", "file_mtime", F.col("doc.cik").alias("cik"),
                           F.trim("doc.entityName").alias("entity_name"), F.explode("doc.facts").alias("taxonomy", "cs"))
                   .select("*", F.explode("cs").alias("concept", "cv")).drop("cs"))

flat = (concepts_df
        .select("*", F.explode_outer("cv.units").alias("unit", "arr"))
        .select("*", F.posexplode_outer("arr").alias("pos", "e"))
        .select("source_file", F.col("sha256").alias("source_file_sha256"), F.col("file_mtime").alias("source_file_mtime"),
                "cik", F.lpad(F.col("cik").cast("string"), 10, "0").alias("cik_padded"), "entity_name",
                "taxonomy", "concept", F.trim("cv.label").alias("concept_label"), "unit",
                F.coalesce(unit_map_expr[F.col("unit")], F.col("unit")).alias("unit_normalized"),
                F.col("e.start").alias("raw_start"), F.col("e.end").alias("raw_end"), F.col("e.filed").alias("raw_filed"),
                F.expr("try_to_date(e.start, 'yyyy-MM-dd')").alias("period_start"),
                F.expr("try_to_date(e.end, 'yyyy-MM-dd')").alias("period_end"),
                F.expr("try_to_date(e.filed, 'yyyy-MM-dd')").alias("filed_date"),
                F.col("e.val").alias("value"), F.trim("e.accn").alias("accession_number"), F.trim("e.form").alias("form"),
                F.col("e.fy").alias("filing_fiscal_year"), F.upper(F.trim("e.fp")).alias("filing_fiscal_period"),
                F.nullif(F.trim("e.frame"), F.lit("")).alias("frame"), F.to_json("e").alias("raw_fact"))
        .withColumn("period_type", F.when(F.col("raw_start").isNull(), "instant").otherwise("duration"))
        .withColumn("duration_days", F.when(F.col("period_start").isNotNull(), F.datediff("period_end", "period_start") + 1))
        .withColumn("is_amendment", F.coalesce(F.col("form").endswith("/A"), F.lit(False))))

# independent expected count: sum of array sizes per file (not via explode) - used for reconciliation in step 10
expected_counts = (concepts_df.select("cik", F.explode("cv.units").alias("u", "arr"))
                   .groupBy("cik").agg(F.sum(F.size("arr")).alias("expected_facts")))
print("columns:", flat.columns)

In [0]:
# Step 7: hard rules -> quarantine (row rejected, kept for review); soft rules -> dq_warnings array (row kept, flagged)
HARD = {
    "NULL_CIK": "cik IS NULL",
    "NULL_CONCEPT_OR_UNIT": "concept IS NULL OR unit IS NULL",
    "EMPTY_FACT": "raw_fact IS NULL",
    "INVALID_PERIOD_END": "period_end IS NULL",
    "INVALID_PERIOD_START": "raw_start IS NOT NULL AND period_start IS NULL",
    "START_AFTER_END": "period_start > period_end",
    "NULL_VALUE": "value IS NULL",
    "INVALID_ACCESSION": "accession_number IS NULL OR NOT accession_number RLIKE '^[0-9]{10}-[0-9]{2}-[0-9]{6}$'",
    "INVALID_FILED_DATE": "filed_date IS NULL OR filed_date > current_date()",
    "NULL_FORM": "form IS NULL OR form = ''",
}
forms_sql = ",".join(f"'{x}'" for x in PERIODIC_FORMS)
SOFT = {
    "FILED_BEFORE_PERIOD_END": "filed_date < period_end",
    "MISSING_CONCEPT_LABEL": "concept_label IS NULL",
    "PERIODIC_FORM_MISSING_FY_FP": f"form IN ({forms_sql}) AND (filing_fiscal_year IS NULL OR filing_fiscal_period IS NULL)",
    "INVALID_FISCAL_PERIOD": "filing_fiscal_period IS NOT NULL AND filing_fiscal_period NOT IN ('FY','Q1','Q2','Q3','Q4','H1','H2')",
    "UNUSUAL_DURATION": "duration_days IS NOT NULL AND (duration_days < 28 OR duration_days > 400)",
    "PERIOD_BEFORE_1993": "period_end < DATE'1993-01-01'",
    "UNIT_NORMALIZED": "unit <> unit_normalized",
}
def rule_array(rules):
    return F.filter(F.array(*[F.when(F.expr(f"coalesce({c}, false)"), F.lit(n)) for n, c in rules.items()]), lambda x: x.isNotNull())

checked = flat.withColumn("dq_errors", rule_array(HARD)).withColumn("dq_warnings", rule_array(SOFT))

# materialise once as a run-scoped staging table (deterministic, avoids re-reading/re-parsing bronze for every action)
STG = f"{NS}._stg_company_facts"
checked.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STG)
checked = spark.table(STG)

valid  = checked.filter(F.size("dq_errors") == 0)
quar   = checked.filter(F.size("dq_errors") > 0)
ROWS_BRONZE, ROWS_QUAR = checked.count(), quar.count()

(quar.select(F.lit(RUN_ID).alias("run_id"), "source_file", "cik", "taxonomy", "concept", "unit", "raw_fact", "dq_errors",
             F.current_timestamp().alias("quarantined_at")).write.mode("append").saveAsTable(T_QUAR))

agg = checked.agg(*[F.count_if(F.array_contains("dq_errors", n)).alias(n) for n in HARD],
                  *[F.count_if(F.array_contains("dq_warnings", n)).alias(n) for n in SOFT]).first().asDict()
dq_rows = [(RUN_ID, n, "ERROR" if n in HARD else "WARN", int(v), ROWS_BRONZE) for n, v in agg.items()]
(spark.createDataFrame(dq_rows, "run_id string, rule string, severity string, failed_rows bigint, total_rows bigint")
      .withColumn("checked_at", F.current_timestamp()).write.mode("append").saveAsTable(T_DQ))
print(f"bronze facts={ROWS_BRONZE:,}  quarantined={ROWS_QUAR:,}")
display(spark.createDataFrame(dq_rows, "run_id string, rule string, severity string, failed_rows bigint, total_rows bigint").drop("run_id"))

In [0]:
# Step 8: natural key = cik|taxonomy|concept|unit|start|end|accn (one value per filing per period).
# Deterministic dedup, record_hash for change detection, is_latest_filing for 'as currently reported' gold views.
KEY = ["cik", "taxonomy", "concept", "unit", "period_start", "period_end", "accession_number"]
key_str = F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("~")) for c in KEY])

w_dup = Window.partitionBy(*KEY).orderBy(F.col("frame").isNull(), F.desc("filed_date"), F.desc("source_file_mtime"), "raw_fact")
dedup = (valid.withColumn("fact_id", F.sha2(key_str, 256))
              .withColumn("_rn", F.row_number().over(w_dup)))
conflicts = dedup.groupBy("fact_id").agg(F.countDistinct("value").alias("n")).filter("n > 1").count()
dedup = dedup.filter("_rn = 1").drop("_rn")

w_latest = Window.partitionBy("cik", "taxonomy", "concept", "unit", "period_start", "period_end").orderBy(F.desc("filed_date"), F.desc("accession_number"))
HASH_COLS = ["entity_name", "concept_label", "unit_normalized", "value", "form", "filing_fiscal_year", "filing_fiscal_period", "filed_date", "frame"]
silver_batch = (dedup
    .withColumn("is_latest_filing", F.row_number().over(w_latest) == 1)
    .withColumn("record_hash", F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("~")) for c in HASH_COLS], F.col("is_latest_filing").cast("string"), F.array_join("dq_warnings", ",")), 256))
    .select("fact_id", "cik", "cik_padded", "entity_name", "taxonomy", "concept", "concept_label", "unit", "unit_normalized",
            "period_type", "period_start", "period_end", "duration_days", "value", "accession_number", "form", "is_amendment",
            "filing_fiscal_year", "filing_fiscal_period", "filed_date", "frame", "is_latest_filing", "dq_warnings", "record_hash",
            "source_file", "source_file_sha256", "source_file_mtime"))

ROWS_VALID = valid.count()
ROWS_BATCH = silver_batch.count()
ROWS_DUPES = ROWS_VALID - ROWS_BATCH
assert conflicts == 0, f"{conflicts} natural keys carry conflicting values - investigate before loading"
assert silver_batch.select("fact_id").distinct().count() == ROWS_BATCH, "fact_id not unique after dedup"
print(f"valid={ROWS_VALID:,} duplicates_removed={ROWS_DUPES:,} batch={ROWS_BATCH:,} conflicting_keys={conflicts}")
display(silver_batch.groupBy("is_latest_filing", "period_type").count())

In [0]:
# Step 9: MERGE on fact_id. Update only when record_hash changes (idempotent re-runs = 0 writes).
# Each file is a FULL snapshot of a CIK, so facts that vanished for a processed CIK are soft-deleted (is_active=false), never hard-deleted.
batch_ciks = [r.cik for r in silver_batch.select("cik").distinct().collect()]
now = F.current_timestamp()
src = silver_batch.withColumn("_run", F.lit(RUN_ID))
cik_list = ",".join(str(c) for c in batch_ciks) or "NULL"

upd_cols = {c: f"s.{c}" for c in silver_batch.columns if c != "fact_id"}
upd_cols.update({"is_active": "true", "last_updated_run_id": "s._run", "_updated_at": "current_timestamp()"})
ins_cols = dict(upd_cols, fact_id="s.fact_id", first_loaded_run_id="s._run", _created_at="current_timestamp()")

(DeltaTable.forName(spark, T_FACTS).alias("t")
   .merge(src.alias("s"), "t.fact_id = s.fact_id")
   .whenMatchedUpdate(condition="t.record_hash <> s.record_hash OR t.is_active = false", set=upd_cols)
   .whenNotMatchedInsert(values=ins_cols)
   .whenNotMatchedBySourceUpdate(condition=f"t.cik IN ({cik_list}) AND t.is_active = true",
                                 set={"is_active": "false", "last_updated_run_id": f"'{RUN_ID}'", "_updated_at": "current_timestamp()"})
   .execute())
m = spark.sql(f"DESCRIBE HISTORY {T_FACTS} LIMIT 1").first().operationMetrics
ROWS_INS, ROWS_UPD = int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsMatchedUpdated", 0))
ROWS_DEL = int(m.get("numTargetRowsNotMatchedBySourceUpdated", 0))

# company dimension (SCD1) - built from the merged silver facts
company_src = (spark.table(T_FACTS).filter(F.col("is_active") & F.col("cik").isin(batch_ciks))
   .groupBy("cik", "cik_padded").agg(F.max_by("entity_name", "filed_date").alias("entity_name"),
        F.array_sort(F.collect_set("taxonomy")).alias("taxonomies"), F.countDistinct("taxonomy", "concept").cast("int").alias("concept_count"),
        F.count("*").alias("fact_count"), F.min("period_end").alias("first_period_end"), F.max("period_end").alias("last_period_end"),
        F.max("filed_date").alias("last_filed_date"), F.max("source_file").alias("source_file"), F.max("source_file_sha256").alias("source_file_sha256"))
   .withColumn("last_updated_run_id", F.lit(RUN_ID)).withColumn("_updated_at", now))
(DeltaTable.forName(spark, T_COMPANY).alias("t").merge(company_src.alias("s"), "t.cik = s.cik")
   .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

# concept dictionary (SCD1) - one row per taxonomy+concept across all companies
concept_src = (concepts_df.select("taxonomy", "concept", F.trim("cv.label").alias("label"), F.trim("cv.description").alias("description"), F.map_keys("cv.units").alias("u"))
   .groupBy("taxonomy", "concept").agg(F.max("label").alias("label"), F.max("description").alias("description"),
        F.array_sort(F.array_distinct(F.flatten(F.collect_list("u")))).alias("units"))
   .withColumn("last_updated_run_id", F.lit(RUN_ID)).withColumn("_updated_at", now))
(DeltaTable.forName(spark, T_CONCEPT).alias("t").merge(concept_src.alias("s"), "t.taxonomy = s.taxonomy AND t.concept = s.concept")
   .whenMatchedUpdate(set={"label": "coalesce(s.label, t.label)", "description": "coalesce(s.description, t.description)",
                           "units": "array_sort(array_distinct(concat(t.units, s.units)))", "last_updated_run_id": "s.last_updated_run_id", "_updated_at": "s._updated_at"})
   .whenNotMatchedInsertAll().execute())
print(f"facts inserted={ROWS_INS:,} updated={ROWS_UPD:,} soft_deleted={ROWS_DEL:,}")

In [0]:
# Step 10: prove completeness with an independent count, close the audit record, update the manifest, fail loudly on mismatch
recon = (expected_counts.alias("e")
   .join(checked.groupBy("cik").agg(F.count("*").alias("flattened"), F.count_if(F.size("dq_errors") > 0).alias("quarantined")), "cik", "full")
   .join(silver_batch.groupBy("cik").agg(F.count("*").alias("batch_rows")), "cik", "full")
   .join(spark.table(T_FACTS).filter("is_active").groupBy("cik").agg(F.count("*").alias("silver_active")), "cik", "left")
   .withColumn("bronze_eq_flattened", F.col("expected_facts") == F.col("flattened"))
   .withColumn("silver_eq_batch", F.col("silver_active") == F.col("batch_rows"))
   .orderBy("cik"))
recon_pd = recon.toPandas()
RECON_OK = bool(len(recon_pd) == FILES_TO_PROCESS - FILES_FAILED and recon_pd.bronze_eq_flattened.all() and recon_pd.silver_eq_batch.all()) if len(recon_pd) else FILES_TO_PROCESS == 0
display(recon_pd)

# manifest: one ledger row per discovered file
fs = file_status.assign(status=["FAILED_PARSE" if e else "PROCESSED" for e in file_status.parse_error])
cnt = dict(zip(recon_pd.get("cik", []), recon_pd.get("expected_facts", [])))
fs["fact_count"] = [int(cnt.get(int(c), 0)) for c in fs.cik_from_path]
man = spark.createDataFrame(fs[["source_file", "file_size", "file_mtime", "sha256", "cik_from_path", "status", "fact_count"]]) if len(fs) else None
if FILES_SKIPPED:
    sk = skipped.withColumn("status", F.lit("SKIPPED_UNCHANGED")).withColumn("fact_count", F.lit(None).cast("bigint"))
    man = sk if man is None else man.unionByName(sk)
if man is not None:
    (man.withColumn("file_size", F.col("file_size").cast("bigint")).withColumn("file_mtime", F.col("file_mtime").cast("timestamp"))
        .withColumn("run_id", F.lit(RUN_ID)).withColumn("processed_at", F.current_timestamp())
        .write.mode("append").saveAsTable(T_MANIFEST))

status = "SUCCESS" if RECON_OK and FILES_FAILED == 0 else ("PARTIAL" if RECON_OK else "FAILED_RECONCILIATION")
err = None if status == "SUCCESS" else f"files_failed={FILES_FAILED}; recon_ok={RECON_OK}"
spark.sql(f"""UPDATE {T_AUDIT} SET run_end = current_timestamp(), status = '{status}', files_discovered = {FILES_DISCOVERED},
  files_processed = {FILES_TO_PROCESS - FILES_FAILED}, files_skipped = {FILES_SKIPPED}, files_failed = {FILES_FAILED},
  rows_bronze = {ROWS_BRONZE}, rows_valid = {ROWS_VALID}, rows_quarantined = {ROWS_QUAR}, rows_duplicates_removed = {ROWS_DUPES},
  rows_inserted = {ROWS_INS}, rows_updated = {ROWS_UPD}, rows_soft_deleted = {ROWS_DEL}, reconciliation_passed = {RECON_OK},
  error_message = {'NULL' if err is None else repr(err)} WHERE run_id = '{RUN_ID}'""")

spark.sql(f"DROP TABLE IF EXISTS {STG}")  # run-scoped staging only
spark.sql(f"OPTIMIZE {T_FACTS}")
display(spark.table(T_AUDIT).filter(F.col("run_id") == RUN_ID))
if not RECON_OK:
    raise Exception(f"Reconciliation failed for run {RUN_ID} - see {T_AUDIT}")